In [ ]:
# 1. Create the Gold Fact Sales Table
# We join Silver Sales with Gold Product to get the 'Marked Price' for Discount logic
spark.sql("""
CREATE OR REPLACE TABLE de_mini_project.gold.fact_sales AS
WITH sales_enriched AS (
    SELECT 
        s.transaction_id,
        s.date,
        s.product_id,
        s.customer_id,
        s.store_id,
        s.sold_qty,
        s.unit_price AS actual_sold_price,
        p.selling_price AS original_marked_price,
        -- KPI #1 & #4: Revenue and Quantity base
        (s.sold_qty * s.unit_price) AS total_revenue,
        -- KPI #8: Discount Impact (Difference between what we wanted to charge vs what we got)
        (p.selling_price - s.unit_price) * s.sold_qty AS discount_loss_amount,
        -- KPI #5 Prep: Check if this transaction exists in the returns table
        CASE WHEN r.transaction_id IS NOT NULL THEN 1 ELSE 0 END AS is_returned
    FROM de_mini_project.silver.sales s
    LEFT JOIN de_mini_project.gold.dim_product p ON s.product_id = p.product_id
    LEFT JOIN de_mini_project.silver.returns r ON s.transaction_id = r.transaction_id
)
SELECT * FROM sales_enriched
""")

# 2. Validation Check
# Let's see if we captured the Discount Impact correctly
display(spark.sql("""
    SELECT transaction_id, actual_sold_price, original_marked_price, discount_loss_amount 
    FROM de_mini_project.gold.fact_sales 
    WHERE discount_loss_amount > 0 
    LIMIT 5
"""))